# Semantic Query Processing with DocETL

In a database course, we use SQL to filter rows, join tables, and aggregate values by grouping keys. This works well when the information we need is already stored in columns.

Company reports, emails, and customer reviews contain useful information, but it is often mixed with other details in unstructured text. To analyze these documents, people often have to read them, find the relevant information, and organize it. Language models can help with this work.

**Semantic query processing** lets us describe these tasks using database operators such as filtering, joining, and aggregation, with LLMs interpreting the text within these operations. For example, a filter can ask whether a document reports a company’s environmental work, rather than simply checking whether it contains the word “environment.” We specify what we want; the system organizes the model calls and can explore different ways to run the query.

We will explore this approach with **DocETL**, a system for querying documents with language models. First, we will try operators like `filter`, `map`, `equijoin`, and `reduce`, then combine them into a query. We will also use **MOAR**, DocETL’s query optimizer, to see how it changes a query and what happens to the results.

## 1. Look at the Documents

We selected six excerpts from the [DataNeed company-report collection](https://huggingface.co/datasets/DataNeed/company-reports), which contains over 5,000 records. They include sustainability reports, a financial announcement, and a job ad. We want to compare companies' reported environmental work, so we first need to find their own reports.

The excerpts are in `document_data/reports/`; omission markers indicate skipped passages. The next cells load the data and display two examples.


Start this lab in a new Colab runtime. Keep personal files and unrelated API keys out of it, and use the same runtime throughout.

Run the next cell to download the six excerpts. You can browse them in Colab's Files panel under `runs/document_course/document_data/reports/`; we will read two examples below.


In [ ]:
#@title Download the example files
from pathlib import Path
import hashlib
import sys
import urllib.error
import urllib.request
import zipfile

IN_COLAB = 'google.colab' in sys.modules
bundle = Path('document_lab_setup.zip')
COURSE_BUNDLE_URL = (
    'https://raw.githubusercontent.com/SleepyLGod/AIST4020-Lab-Tutorials/'
    '3df1e011cbfd41fb667818b69903473aa5ca4f97/'
    'labs/docetl/semantic-query-processing/document_lab_setup.zip'
)
COURSE_BUNDLE_SHA256 = '8487710956aab4e6db5560ab86f1de2250f34c82e826e5a4264295e96ed56870'

if not bundle.exists():
    if not COURSE_BUNDLE_URL.startswith('https://'):
        raise ValueError('The course download must use HTTPS.')
    print('Downloading the course files...')
    try:
        with urllib.request.urlopen(COURSE_BUNDLE_URL, timeout=60) as response:
            bundle_bytes = response.read(1_000_001)
    except urllib.error.URLError as exc:
        raise RuntimeError('The course download failed. Check your connection or notify your instructor.') from exc
    if len(bundle_bytes) > 1_000_000 or hashlib.sha256(bundle_bytes).hexdigest() != COURSE_BUNDLE_SHA256:
        raise ValueError('The course download does not match this notebook version.')
    bundle.write_bytes(bundle_bytes)
if hashlib.sha256(bundle.read_bytes()).hexdigest() != COURSE_BUNDLE_SHA256:
    raise ValueError('The course bundle does not match this notebook version.')
COURSE_DIR = Path('runs/document_course')
with zipfile.ZipFile(bundle) as archive:
    archive.extractall(COURSE_DIR)  # Only the exact, checksum-verified course bundle.
sys.path.insert(0, str(COURSE_DIR.resolve()))
import document_lab_support as support
DATA_DIR = COURSE_DIR / 'document_data'
documents, sector_rows = support.load_data(DATA_DIR)
print('Documents:', ', '.join(path.name for path in sorted((DATA_DIR / 'reports').glob('*.txt'))))


In [ ]:
support.show_document(documents, 'R05')
support.show_document(documents, 'R10')


XPO (`R05.txt`) describes its own environmental measures, including installing LED lighting. Sustainserv (`R10.txt`) is advertising a job writing sustainability reports. Both mention sustainability, but only XPO's excerpt reports the kind of activity we want to compare.

Each input record has two fields: `doc_id` identifies the excerpt, and `text` contains its words.

You can open all six `.txt` files in Colab's Files panel. The source links below point to the original documents.


In [ ]:
import json
sources = json.loads((DATA_DIR / 'provenance.json').read_text())
support.show_rows(sources, ['doc_id', 'original_filename', 'original_url'])


## 2. Query the Documents

We will produce one row per sector (industry), with a company count and a summary of reported environmental measures. This is a small analytical workload of the kind studied in **OLAP** (online analytical processing): select relevant records, relate them to categories, and summarize by group.

### Choose One Model

Keep `backend = 'deepseek'`. **Set `ENABLE_MODEL_CALLS = True`**, then run the installation cell below and enter your API key when asked. Input is hidden; an existing environment variable or Colab Secret is used first. In Colab, allow access to the selected Secret.

**Other options:** choose Ollama for a local model (a GPU is recommended), NVIDIA, or `custom` for your own account. For custom, fill in the model and its key environment-variable name, not the key itself. Speed and response formats vary by model.

DocETL uses LiteLLM model names such as `deepseek/deepseek-flash`. Hosted calls may incur charges; each query can make several requests.


In [ ]:
BACKENDS = {
    'deepseek': {'model': 'deepseek/deepseek-flash', 'key_env': 'DEEPSEEK_API_KEY'},
    'ollama': {'model': 'ollama_chat/qwen3.5:9b-q4_K_M', 'key_env': ''},
    'nvidia': {'model': 'nvidia_nim/nvidia/nemotron-3-super-120b-a12b', 'key_env': 'NVIDIA_NIM_API_KEY'},
    'custom': {'model': '', 'key_env': ''},
}
backend = 'deepseek'
ENABLE_MODEL_CALLS = False


In [ ]:
import lab_support
RUN_DIR = Path('runs/document_queries')
RUN_DIR.mkdir(parents=True, exist_ok=True)
versions = lab_support.install_docetl(RUN_DIR)
import docetl
shell = get_ipython() if 'get_ipython' in globals() else None
if shell is not None:
    shell.run_line_magic('matplotlib', 'inline')
MODEL_REF, OPTIONS = support.setup_model(backend, BACKENDS, IN_COLAB, RUN_DIR)
docetl.default_model = MODEL_REF
docetl.max_threads = 1
docs = docetl.from_list(documents, name='documents')
print(versions)


### Filter: Which Documents Should Stay?

We only want reports about a company's own sustainability work, not financial announcements or job ads. ESG in the prompt means environmental, social, and governance.

A `Frame` holds our records. The function below uses `filter` to define which records to keep; `collect()` starts execution. DocETL fills `{{ input.text }}` with one excerpt, and the model returns `keep`: true keeps the record, false removes it.


In [ ]:
def keep_reports(frame: docetl.Frame) -> docetl.Frame:
    """Describe which document records to retain."""
    return frame.filter(
        name='keep_sustainability_reports',
        prompt="""Is this excerpt from a report primarily about a company's own
sustainability, ESG, or corporate responsibility work?
Exclude ordinary financial reports, earnings announcements, and job ads.
Excerpt: {{ input.text }}""",
        output={'schema': {'keep': 'bool'}},
        litellm_completion_kwargs=OPTIONS,
    )

support.require_enabled(ENABLE_MODEL_CALLS)
kept_rows = keep_reports(docs).collect()
support.check_rows(documents, kept_rows, complete=False)
support.show_rows(kept_rows, ['doc_id', 'text'])


Check whether XPO (`R05.txt`) stayed and the Sustainserv job ad (`R10.txt`) disappeared. You can also inspect `R08.txt`: "sustainable growth" in a financial announcement need not mean environmental action.

This meaning-based true/false condition is a **semantic predicate**. The result contains the retained records; DocETL uses and then removes the temporary `keep` field.

The key lines in [DocETL's filter implementation](https://github.com/ucbepic/docetl/blob/0.3.0/docetl/operations/filter.py) make that last step visible:

```python
keep_record = bool(result.get(self._filter_key))
result.pop(self._filter_key, None)
```

`self._filter_key` is `keep` in our query. [Filter API](https://ucbepic.github.io/docetl/operators/filter/)

### Map: Turn Text into Fields

Next, turn each retained excerpt into a company record: extract its company name, main business, and up to two environmental measures. Include a quote for each measure so we can check it. The business description will help us match the company to a sector.

The `schema` specifies the fields to return: `string` means text, and `list[string]` means a list of text items.

Here is a format example for one `actions` item, not a model result:

- **Source:** “We’re installing LED lighting in our facilities”
- **→ `actions` item:** LED lighting installation; ongoing; quote: “We’re installing LED lighting in our facilities”

The measure, status, and quote go together in one text item.


In [ ]:
def extract_actions(frame: docetl.Frame) -> docetl.Frame:
    """Describe the company facts to extract from each record."""
    return frame.map(
        name='extract_company_actions',
        prompt="""Read this report excerpt: {{ input.text }}
Extract the company name, its main business, and up to two concrete
environmental actions or commitments, one measure per list item. Preserve
whether work is completed, ongoing, or planned. For example, 'is installing'
means ongoing, not completed. If the status or actor is unclear, do not infer it. For each action include a short exact quote from the excerpt.
If an item is not stated, say 'not stated'; do not add outside facts.""",
        output={'schema': {
            'company': 'string', 'business': 'string', 'actions': 'list[string]',
        }},
        litellm_completion_kwargs=OPTIONS,
    )

support.require_enabled(ENABLE_MODEL_CALLS)
fact_rows = extract_actions(docetl.from_list(kept_rows, name='kept_reports')).collect()
support.check_rows(kept_rows, fact_rows, complete=True)
support.show_rows(fact_rows, ['doc_id', 'company', 'business', 'actions'])


Check one extracted measure against its quote. If XPO's result includes LED lighting, compare it with `R05.txt`: "we're installing" should remain work in progress, not a completed project.

`map` adds the requested fields to each record, keeping the original `doc_id` and `text`. Unlike the filter prompt, this prompt asks **what to extract**, rather than whether to keep the record. [Map API](https://ucbepic.github.io/docetl/operators/map/)

### Join: Match Businesses to Sector Definitions

To compare sectors, we need a sector name on each company row. Load a second table with two sector definitions, written for this lab: manufacturing and transportation/logistics. We will match each company's business description to these definitions.


In [ ]:
support.show_rows(sector_rows, ['sector', 'definition'])
sectors = docetl.from_list(sector_rows, name='sectors')


For each company and each definition, ask whether they match. Boeing (`R02.txt`) is a useful test: making aircraft is different from operating a transport service. The condition must compare the business with the definition, not just look for a word such as "airplane."


In [ ]:
def match_sectors(frame: docetl.Frame, definitions: docetl.Frame) -> docetl.Frame:
    """Associate company records with matching sector definitions."""
    return frame.equijoin(
        definitions,
        name='match_business_to_sector',
        comparison_prompt="""Business: {{ left.business }}
Sector definition: {{ right.definition }}
Does the company's main business fit this definition?
Use the business described, not an industry suggested by its name.""",
        blocking_conditions=['True'],
        comparison_model=MODEL_REF,
        litellm_completion_kwargs=OPTIONS,
    )

support.require_enabled(ENABLE_MODEL_CALLS)
matched_rows = match_sectors(docetl.from_list(fact_rows, name='company_facts'), sectors).collect()
support.inspect_matches(fact_rows, sector_rows, matched_rows)
support.show_rows(matched_rows, ['doc_id', 'company', 'business', 'sector'])


`left.business` comes from the company table; `right.definition` comes from the sector table. A match produces one combined row. DocETL calls this API `equijoin`, but here the prompt defines a semantic condition, not string equality.

`blocking_conditions=['True']` lets every company try every definition. Four companies and two definitions give eight candidate pairs. Check any company with zero or multiple matches before reading the summary. [Equijoin API](https://ucbepic.github.io/docetl/operators/equijoin/)

### Reduce: Summarize Each Sector

We now have company rows with sector names. Group them by `sector` and ask the model to summarize each group's actions. For transportation/logistics, the summary should distinguish XPO's measures from Tallink's. Tallink's shore power refers to electricity supplied from land while a vessel is docked.

The **model writes the summary; Python counts the companies** from the saved input IDs. This subset has one document per company.


In [ ]:
def summarize_sectors(frame: docetl.Frame) -> docetl.Frame:
    """Describe one summary for each sector group."""
    return frame.reduce(
        name='summarize_by_sector',
        reduce_key='sector',
        prompt="""Summarize the reported actions in this sector.
{% for item in inputs %}
Company: {{ item.company }}; actions: {{ item.actions }}
{% endfor %}
Keep the company names and preserve whether each measure is completed,
ongoing, or planned. Do not turn ongoing work into a completed action.
If a status is unclear, leave it unclear. Use only the supplied records.
Return the summary as a text string, not a nested object.""",
        output={'schema': {'summary': 'string'}, 'lineage': ['doc_id']},  # Save source IDs for counting.
        synthesize_resolve=False,  # Sector names are already standardized.
        litellm_completion_kwargs=OPTIONS,
    )

support.require_enabled(ENABLE_MODEL_CALLS)
summary_rows = summarize_sectors(docetl.from_list(matched_rows, name='matched_companies')).collect()
support.check_summaries(matched_rows, summary_rows)
analysis_rows = support.add_company_counts(summary_rows)
support.show_rows(analysis_rows, ['sector', 'company_count', 'summary'])


`reduce_key='sector'` works like `GROUP BY sector`: equal sector names form a group. Grouping does not require a model. The prompt asks the model to summarize the group's text; `inputs` holds those records, and the template loop inserts their actions. [Reduce API](https://ucbepic.github.io/docetl/operators/reduce/)

Compare one summary with its company rows: whose measures are described, and do ongoing projects still read as ongoing? Check the count against the same rows.

## 3. Put the Operations Together

Follow XPO through the results already produced. Each row below shows what one operation did to the same document. If a step lost it or matched it twice, that is part of this run's result.

Then reuse the four functions to build the complete query.


In [ ]:
support.show_trace(documents, kept_rows, fact_rows, matched_rows, summary_rows)


In [ ]:
query = keep_reports(docs)
query = extract_actions(query)
query = match_sectors(query, sectors)
query = summarize_sectors(query)

plan_figure = support.plot_plan()


These four lines build a **query plan**. We chose the steps and their order; DocETL executes them when we call `collect()`. Later, MOAR will try changing a query plan.

| Operation | Question | Change to the records |
| --- | --- | --- |
| `filter` | Is this a company's sustainability report? | Keep relevant documents. |
| `map` | What company, business, and measures does it describe? | Add fields from the text. |
| `equijoin` | Which sector definition fits the business? | Attach matching sector records. |
| `reduce` | What do companies in each sector report? | Produce one summary per group. |

The next cell optionally runs the whole query again. Leave it off to use the stage-by-stage results without additional model calls.


In [ ]:
RUN_COMPLETE_QUERY = False
if RUN_COMPLETE_QUERY:
    support.require_enabled(ENABLE_MODEL_CALLS)
    complete_rows = query.collect()
    complete_analysis_rows = support.add_company_counts(complete_rows)
    support.show_rows(complete_analysis_rows, ['sector', 'company_count', 'summary'])
else:
    print('Plan built. The stage-by-stage result is shown above; no additional query was run.')


## 4. Count Reports by Sector

A summary can have several reasonable wordings. Before trying an optimization, let’s set the summary aside and **select the reports, assign sectors, and count the companies**. We can check these decisions document by document.

```text
Six documents → filter: keep reports → map: assign sector → Python: count rows
```

There are only two sectors here, so we can put their definitions in a `map` prompt. Instead of using a join, the model reads each report and fills in `sector`.


In [ ]:
def assign_sector(frame: docetl.Frame) -> docetl.Frame:
    """Add a sector using the two teaching definitions."""
    definitions = "\n".join(
        f"- {row['sector']}: {row['definition']}" for row in sector_rows
    )
    return frame.map(
        name='classify_report_sector',
        prompt=(
            "Read this company report excerpt:\n{{ input.text }}\n\n"
            "Classify the company's main business using these definitions:\n"
            + definitions
            + "\nReturn exactly one of the sector names above in sector. "
              "Use the main business described, not the environmental action "
              "or the industry of its customers."
        ),
        output={'schema': {'sector': 'string'}},
        litellm_completion_kwargs=OPTIONS,
    )

sector_query = assign_sector(
    keep_reports(docetl.from_list(documents, name='sector_documents'))
)


### Run It and Check Each Document

The next cell runs the two-step query and shows its decisions and counts. Compare each decision with the source-checked reference:

| Document | Expected result |
| --- | --- |
| R01, R02 | Manufacturing |
| R05, R06 | Transportation and logistics |
| R08, R10 | Excluded |

**Accuracy = correct decisions / 6.** For example, missing R06 while getting the others right gives 5/6. We check documents, not just totals: swapping two companies' sectors could leave both counts unchanged.

Python reads the reference for scoring; the model does not receive it. These six excerpts are our visible classroom examples, not a representative sample or an independent test set.


In [ ]:
support.require_enabled(ENABLE_MODEL_CALLS)
sector_baseline_rows, baseline_work = support.collect_measured(sector_query)
reference = support.load_reference(DATA_DIR)
support.show_sector_result(sector_baseline_rows, reference)
support.show_query_work([{'id': 'original', **baseline_work}])


## 5. Let MOAR Try a Different Plan

Our query may read a report twice: once to decide whether to keep it, and again to assign a sector. Could another plan reduce that work or make better judgments?

**MOAR starts from our original query (the baseline), proposes a new plan (a candidate), runs it, and checks its score.** Python calculates the same six-document accuracy for each plan.

| Optimization | What changes | Example to consider |
| --- | --- | --- |
| **Logical Plan Rewriting** | How operations are arranged | Combine steps that inspect the same document. |
| **Physical Execution Optimization** | How an operation is performed | Rewrite the sector-classification prompt. |

Ordinary SQL optimization normally preserves the query result. Changing a model's instructions can change its answers, so here we also measure quality. Some rewrites affect both the steps and their implementation.

The next cell calls `Frame.optimize(...)` for a small search using our two-step query, the same model, and all six documents. **This can make several paid requests and execute code generated by the optimizer.** Set `RUN_MOAR = True` to start.

MOAR can also try retrieving relevant passages, so the first search prepares a small local embedding model for text search. Your chosen chat model stays the same.


In [ ]:
RUN_MOAR = False
ALLOW_LOCAL_EMBEDDINGS = True  # Allow local passage retrieval with nomic-embed-text:v1.5.
search_record = None
if RUN_MOAR:
    support.require_enabled(ENABLE_MODEL_CALLS)
    search_record = support.run_optimization(
        sector_query, documents, sector_baseline_rows,
        reference, MODEL_REF, RUN_DIR,
        enabled=True, in_colab=IN_COLAB,
        allow_local_embeddings=ALLOW_LOCAL_EMBEDDINGS, baseline_work=baseline_work,
    )
support.show_search(search_record, documents)


### Read One Change, Then Compare the Plans

Use your output to follow three questions:

1. **What changed?** Compare the old and new instructions. If a document's decision changed, read its text to see which answer it supports. A longer prompt is not necessarily better.
2. **Did more answers become correct?** Check `correct_documents` and `accuracy`. If the original already got 6/6, accuracy cannot improve on these six documents. The next question is whether a new plan can keep those answers while doing less work.
3. **Did the work decrease?** Compare time, recorded usage, and estimated cost. Unchanged answers with more input tokens may mean extra work without a quality gain.

Here we select a new plan only when accuracy improves; on a tie, we keep the original. An equally accurate but cheaper plan may still be useful. A completed search can therefore leave the original selected.

For a closer cost and time comparison, look in **Run details and errors** for an `unchanged query rerun`: like the candidates, it runs without the query cache. Use it when available rather than the potentially cached `original` row. One run's timing is only an observation; model answers can vary too. Costs are estimates, missing values remain unknown, and search cost is separate from running a query.


In [ ]:
search_figure = support.plot_search(search_record)
if search_record is not None:
    print('Search wall time (seconds):', search_record['search_seconds'])
    print('MOAR SDK search-cost estimate (USD; unverified):',
          search_record['sdk_search_cost_estimate_usd'])
    selected_sector_rows = support.selected_rows(search_record)
    support.show_sector_result(selected_sector_rows, reference)


## 6. Look Inside Prompt Rewriting

The search may choose different strategies. Here we focus on one: asking the model to rewrite the filter prompt using the original instructions and sample documents. For our filter, this could mean clarifying the difference between a company's report and a job ad about writing reports.

The optional cell below tries **Prompt Rewriting** on our filter, using the same data and score. This is a separate experiment from the automatic search. Leave it off to avoid extra calls.


In [ ]:
RUN_PROMPT_REWRITE = False
prompt_record = None
if RUN_PROMPT_REWRITE:
    support.require_enabled(ENABLE_MODEL_CALLS)
    prompt_record = support.run_optimization(
        sector_query, documents, sector_baseline_rows,
        reference, MODEL_REF, RUN_DIR,
        enabled=True, in_colab=IN_COLAB, prompt_only=True, baseline_work=baseline_work,
    )
support.show_search(prompt_record, documents)


These lines from DocETL show how it asks the model to examine the original prompt and sample documents:

```python
f"Original Prompt: {original_prompt}\n\n"
# ...
f"You will be given access to sample input data through a read_next_doc() function. Use this to:\n"
f"1. Understand the actual structure and patterns in the input data\n"
f"2. Identify ambiguities in the original prompt that could be clarified\n"
```

This call asks the model for the new prompt:

```python
schema, updated_message_history, call_cost = runner.run_agentic_loop(
    system_prompt=system_prompt,
    initial_user_message=initial_message,
    response_schema=ClarifyInstructionsInstantiateSchema,
)
# ...
return schema, message_history, call_cost
```

Source: [`clarify_instructions.py`](https://github.com/ucbepic/docetl/blob/0.3.0/docetl/reasoning_optimizer/directives/clarify_instructions.py). `initial_message` carries the request; `schema` contains the proposed `clarified_prompt`, which DocETL puts into the operation. We then run the query to check the effect.

Two other strategies illustrate possible changes, without another experiment. In this version, fusion supports pairs of `map` and `filter` operations, not arbitrary pairs such as `filter` and `code_map`:

| Strategy | Before | After |
| --- | --- | --- |
| **Operator Fusion** | Two model steps read the same document for different fields. | One step returns both fields. |
| **Code Generation / Code Substitution** | A model counts records with known sectors. | Python counts them, as our lab already does. |

See the [fusion](https://github.com/ucbepic/docetl/blob/0.3.0/docetl/reasoning_optimizer/directives/operator_fusion.py) and [code-substitution](https://github.com/ucbepic/docetl/blob/0.3.0/docetl/reasoning_optimizer/directives/swap_with_code.py) implementations.

The prompts express judgments about text; the operators organize the analysis. We run the query, check its answers, and compare alternatives before adopting a change.


### Sources

- [DataNeed company reports](https://huggingface.co/datasets/DataNeed/company-reports): source corpus. The supplied provenance file records the six excerpts and original URLs. They were selected for teaching, not as a representative sample.
- [Semantic Operators](https://arxiv.org/abs/2407.11418): natural-language specifications inside composable data operations.
- [DocETL paper](https://arxiv.org/abs/2410.12189) and [documentation](https://ucbepic.github.io/docetl/): document-processing pipelines and optimization.
- [MOAR paper](https://arxiv.org/abs/2512.02289) and [Python API](https://ucbepic.github.io/docetl/optimization/python-api/): query rewriting and search.
- **Lab scoring reference:** run the cell below to open the bundled scoring notes and reference answers here.
- [DocETL 0.3.0 Frame implementation](https://github.com/ucbepic/docetl/blob/0.3.0/docetl/frame.py): the API used here.

Dataset license declaration: [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). Excerpt sources and omissions are recorded in the supplied provenance file.



In [ ]:
support.show_scoring_reference(COURSE_DIR)
